## **Business Understanding**

In [ ]:
import pandas as pd

# The UCI file is semicolon-separated, not comma-separated — a common gotcha with this dataset
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"
df = pd.read_csv(url, sep=';')

print(df.shape)
df.head()

(4898, 12)


,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


In [2]:
df.info()
df['quality'].value_counts().sort_index()

<class 'pandas.DataFrame'>
RangeIndex: 4898 entries, 0 to 4897
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         4898 non-null   float64
 1   volatile acidity      4898 non-null   float64
 2   citric acid           4898 non-null   float64
 3   residual sugar        4898 non-null   float64
 4   chlorides             4898 non-null   float64
 5   free sulfur dioxide   4898 non-null   float64
 6   total sulfur dioxide  4898 non-null   float64
 7   density               4898 non-null   float64
 8   pH                    4898 non-null   float64
 9   sulphates             4898 non-null   float64
 10  alcohol               4898 non-null   float64
 11  quality               4898 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 459.3 KB


quality
3      20
4     163
5    1457
6    2198
7     880
8     175
9       5
Name: count, dtype: int64

In [3]:
df['good_quality'] = (df['quality'] >= 7).astype(int)

df['good_quality'].value_counts()

good_quality
0    3838
1    1060
Name: count, dtype: int64

In [4]:
X = df.drop(columns=['quality', 'good_quality'])  # all 11 chemical features
y = df['good_quality']                             # our new binary target

In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 80% train, 20% test
    random_state=42,     # makes the split reproducible - same split every time you run it
    stratify=y            # important! keeps the same good/not-good ratio in both sets
)

print(X_train.shape, X_test.shape)

(3918, 11) (980, 11)


In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Scale features — fit on train, apply to both
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train the model
log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train_scaled, y_train)

# Predict on test set
y_pred_log = log_reg.predict(X_test_scaled)

print("Logistic Regression trained!")

Logistic Regression trained!


In [7]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)   # note: unscaled X_train, not X_train_scaled

y_pred_dt = dt.predict(X_test)

print("Decision Tree trained!")

Decision Tree trained!


In [8]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)   # unscaled, same as the Decision Tree

y_pred_rf = rf.predict(X_test)

print("Random Forest trained!")

Random Forest trained!


In [9]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

models = {
    "Logistic Regression": y_pred_log,
    "Decision Tree": y_pred_dt,
    "Random Forest": y_pred_rf
}

for name, y_pred in models.items():
    print(f"--- {name} ---")
    print("Accuracy: ", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:   ", recall_score(y_test, y_pred))
    print("F1 Score: ", f1_score(y_test, y_pred))
    print()

--- Logistic Regression ---
Accuracy:  0.8010204081632653
Precision: 0.5841584158415841
Recall:    0.2783018867924528
F1 Score:  0.3769968051118211

--- Decision Tree ---
Accuracy:  0.8326530612244898
Precision: 0.6008403361344538
Recall:    0.6745283018867925
F1 Score:  0.6355555555555555

--- Random Forest ---
Accuracy:  0.8908163265306123
Precision: 0.8143712574850299
Recall:    0.6415094339622641
F1 Score:  0.7176781002638523



#### Accuracy = % of all predictions that were correct. Easy to understand, but dangerous alone: a model that always predicts "not good" would score ~78% accuracy while catching zero good wines. It's not a bad metric, it's just not sufficient by itself here.

#### Precision = of all the wines the model called "good," how many actually were good? This matters when false positives are costly. Think of it from Powerhouse Motors logic: if you told a customer "this is a premium car" and it wasn't, that damages trust. Same idea here — high precision means when the model says "good wine," you can trust that label.

#### Confusion matrices (seeing it visually)

Numbers aside, let's actually see what's being predicted right/wrong:

In [10]:
for name, y_pred in models.items():
    print(f"--- {name} ---")
    print(confusion_matrix(y_test, y_pred))
    print()

--- Logistic Regression ---
[[726  42]
 [153  59]]

--- Decision Tree ---
[[673  95]
 [ 69 143]]

--- Random Forest ---
[[737  31]
 [ 76 136]]



## Conclusion

### Problem Framing
This assignment applies classification to the UCI White Wine Quality dataset. The original 
`quality` column is a 0–10 taster score, but scores were heavily concentrated around 5–6, with 
very few wines at the extremes (only 20 wines scored 3, and just 5 scored 9). Multi-class 
classification on the raw scores would leave several classes with almost no training examples, 
so the target was converted into a binary label: **good_quality** (1 if quality ≥ 7, else 0). 
This produced a class split of 3,838 "not good" wines vs. 1,060 "good" wines (~78% / 22%) — a 
meaningful class imbalance that shaped every choice below.

### Why Accuracy Alone Isn't Enough
Because of the 78/22 imbalance, a model that always predicted "not good" would score ~78% 
accuracy while being completely useless. Two additional metrics were used to evaluate the models 
properly:

- **Precision** — of all wines a model labeled "good," how many actually were? Low precision 
  means false alarms — wines wrongly flagged as good.
- **Recall** — of all the actual good wines, how many did the model catch? Low recall means 
  missed opportunities — real good wines slipping through undetected.

Both matter here: a winery wants to trust a "good" label (precision) but also doesn't want to 
overlook genuinely good batches (recall). F1 Score, the balance of the two, was tracked as a 
single summary number.

### Results

| Model | Accuracy | Precision | Recall | F1 Score |
|---|---|---|---|---|
| Logistic Regression | 0.801 | 0.584 | 0.278 | 0.377 |
| Decision Tree | 0.833 | 0.601 | 0.675 | 0.636 |
| **Random Forest** | **0.891** | **0.814** | 0.641 | **0.717** |

Logistic Regression struggled most, missing 153 of 212 actual good wines in the test set 
(recall of just 0.278) — a straight-line decision boundary could not separate "good" wines well 
using these chemical features. The Decision Tree improved recall substantially (0.675) but at 
the cost of more false positives (95), reflecting its tendency to overfit to noise in the 
training data. Random Forest struck the best balance: only 31 false positives (highest 
precision) while still correctly identifying 136 of 212 good wines — a direct result of 
averaging predictions across 100 trees, which smooths out the noisy, overfit decisions any 
single tree is prone to.

### Best Model
**Random Forest** is the best-performing model for this task. It leads on accuracy, precision, 
and F1 Score, and trails the Decision Tree only slightly on recall — making it both the most 
reliable and most trustworthy model when flagging a wine as "good."